In [0]:
import requests, json
from datetime import datetime, timedelta

# --- load config (no hardcoding) ---
with open("/lakehouse/default/Files/config/resources.json") as f:
    cfg = json.load(f)

BASE_URL = cfg["base_url"]
RESOURCES = cfg["resources"]
PAGE_COUNT = cfg["page_count"]
LOOKBACK_DAYS = cfg["lookback_days"]

RAW_ROOT = "/lakehouse/default/Files/raw"   # Fabric Lakehouse Files area
# Databricks equivalent: "/dbfs/mnt/raw" or a Unity Catalog volume path area

def fetch_resource(resource_name, since_date):
    """Fetch all pages for one FHIR resource updated since `since_date`."""
    url = f"{BASE_URL}/{resource_name}?_lastUpdated=ge{since_date}&_count={PAGE_COUNT}"
    all_entries = []
    page_num = 0
    call_timestamp = datetime.utcnow().isoformat()

    while url:
        resp = requests.get(url, headers={"Accept": "application/fhir+json"})
        resp.raise_for_status()
        bundle = resp.json()

        entries = bundle.get("entry", [])
        all_entries.extend(entries)

        # save this exact page response, untouched, for full traceability
        save_raw_page(resource_name, since_date, page_num, bundle, call_timestamp, url)

        # follow the "next" link if present, else stop
        next_link = next((l["url"] for l in bundle.get("link", []) if l["relation"] == "next"), None)
        url = next_link
        page_num += 1

    return all_entries

def save_raw_page(resource_name, since_date, page_num, bundle_json, call_timestamp, api_url):
    """Persist the raw API response bucketed by extraction date / resource / page."""
    extraction_date = datetime.utcnow().strftime("%Y-%m-%d")
    folder = f"{RAW_ROOT}/{extraction_date}/{resource_name}"
    dbutils.fs.mkdirs(folder)  # Fabric/Databricks notebook utility

    payload = {
        "resource_type": resource_name,
        "extraction_timestamp": call_timestamp,
        "api_url_or_params": api_url,
        "page_number": page_num,
        "response": bundle_json,
    }
    out_path = f"{folder}/page_{page_num:04d}.json"
    dbutils.fs.put(out_path, json.dumps(payload), overwrite=True)

# --- run for all 4 resources, in dependency order ---
since = (datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")

for resource in RESOURCES:   # order from config: Patient, Encounter, Observation, Condition
    print(f"Fetching {resource} since {since}...")
    entries = fetch_resource(resource, since)
    print(f"  -> {len(entries)} records across pages")